# candidate vocalization identifier demo
~
### run this notebook to see a demonstration of the signal processing pipeline for identifying candidate vocalizations
~
#### what will happen:
#### necessary packages are loaded
#### sequence of signal processing steps are applied to a selected range of the audio
#### plots to visualize of each step of the pipeline will appear
~
#### requirements:
#### keep VocalPy's original directory structure
#### run this notebook from inside the _notebook_ directory
#### have the _audio_example.wav_ directory inside the _audio_ directory (comes with VocalPy by default)

In [ ]:
import os
import sys
import cv2
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (20.0, 6.0)

from PIL import Image
from time import time
from scipy import signal, ndimage
from scipy.io import wavfile
from math import floor, ceil
from skimage import exposure, measure

In [ ]:
def bradley_roth(image, s=None, t=None):
    # -- from somewhere
    img = np.array(image).astype(np.float)

    # -- default window size is round(width/8)
    if s is None:
        s = np.round(img.shape[1]/8)

    # -- default threshold is 15% of the total area in the window
    if t is None:
        t = 15.0

    # -- integral image
    intImage = np.cumsum(np.cumsum(img, axis=1), axis=0)

    # -- define grid of points
    (rows, cols) = img.shape[:2]
    (X, Y)       = np.meshgrid(np.arange(cols), np.arange(rows))

    # -- make into 1D grid of coordinates for easier access
    X = X.ravel()
    Y = Y.ravel()

    # -- ensures is even so that we are able to index the image properly
    s = s + np.mod(s, 2)

    # -- access the four corners of each neighborhood area
    x1 = X - s/2
    x2 = X + s/2
    y1 = Y - s/2
    y2 = Y + s/2

    # -- assert no coordinates are out of bounds
    x1[x1<0]     = 0
    x2[x2>=cols] = cols-1
    y1[y1<0]     = 0
    y2[y2>=rows] = rows-1

    # -- assert coordinates are integers
    x1 = x1.astype(np.int)
    x2 = x2.astype(np.int)
    y1 = y1.astype(np.int)
    y2 = y2.astype(np.int)

    # -- count how many pixels are in each neighborhood
    count = (x2 - x1) * (y2 - y1)

    # -- compute the row and column coordinates to access each corner of the neighborhood for the integral image
    f1_x           = x2
    f1_y           = y2
    f2_x           = x2
    f2_y           = y1 - 1
    f2_y[f2_y < 0] = 0
    f3_x           = x1-1
    f3_x[f3_x < 0] = 0
    f3_y           = y2
    f4_x           = f3_x
    f4_y           = f2_y

    # -- compute areas of each window
    sums = intImage[f1_y, f1_x] - intImage[f2_y, f2_x] - intImage[f3_y, f3_x] + intImage[f4_y, f4_x]

    # -- compute thresholded image and reshape into a 2D grid
    out  = np.zeros(rows*cols, dtype=np.bool)
    out[img.ravel()*count <= sums*(100.0 - t)/100.0] = True

    # -- convert back to uint8
    out  = np.reshape(out, (rows, cols)).astype(np.uint8)

    return out

def imshow_components(labels):
    # Map component labels to hue val
    label_hue = np.uint8(179*labels/np.max(labels))
    blank_ch = 255*np.ones_like(label_hue)
    labeled_img = cv2.merge([label_hue, blank_ch, blank_ch])

    # cvt to BGR for display
    labeled_img = cv2.cvtColor(labeled_img, cv2.COLOR_HSV2BGR)

    # set bg label to black
    labeled_img[label_hue==0] = 0

    plt.imshow(labeled_img)
    plt.show()

In [ ]:
file_path = os.path.join(root_path, 'vocalpy', 'audios', 'audio_example.wav')
print("selected file: {}".format(file_path))

samples, sample_rate = sf.read(file_path)

sample_range = samples[ceil(420 * sample_rate):480*sample_rate]

fs                = sample_rate
window            = signal.get_window('hamming', 1024)
noverlap          = 512
nfft              = 2048
sample_range_secs = sample_range.shape[0] / sample_rate

# -- compute spectrogram
f, t, Pxx = signal.spectrogram(sample_range, fs=fs,
                                             window=window,
                                             noverlap=noverlap,
                                             nfft=nfft,
                                             mode='psd')

# -- apply frequency cutoff
frequency_cutoff = 45000
Pxx = Pxx[(f>frequency_cutoff)]
f = f[(f>frequency_cutoff)]

time_res = sample_range_secs/t.shape[0]
freq_res = (np.max(f) - frequency_cutoff) / f.shape[0]
args = True
# -- convert to dB
Pxx = 10*np.log10(Pxx)

In [ ]:
range_a = 20050
range_b = 20800

if args:
    plt.pcolormesh(t[range_a:range_b], f, Pxx[:,range_a:range_b], cmap='gray')
    plt.title('Spectrogram')
    plt.show()

# -- normalize data
# B = Pxx
# Bmax = np.max(B)
# Bmin = np.min(B)
# B    = (B - Bmin) / (Bmax - Bmin)
B = np.abs(Pxx)
B = B/np.max(B)
if args:
    plt.pcolormesh(t[range_a:range_b], f, B[:,range_a:range_b], cmap='gray')
    plt.title('B normalized')
    plt.show()
    plt.hist(B.ravel(), bins=256, histtype='step', color='black')
    plt.title('B normalized histogram')
    plt.show()

# -- contrast adjustment
p1, p99 = np.percentile(B, (1, 99))
B[B<p1] = 0
B[B>p99]= 1
if args:
    plt.pcolormesh(t[range_a:range_b], f, B[:,range_a:range_b], cmap='gray')
    plt.title('B saturated')
    plt.show()
    plt.hist(B.ravel(), bins=256, histtype='step', color='black')
    plt.title('B saturated histogram')
    plt.show()

# -- binarize spectrogram
B = bradley_roth(B, t=60)
# B = bradley_roth(np.interp(B, (0,255), (0,1)), t=20)
# B = ii8.max - B
if args:
    plt.pcolormesh(t[range_a:range_b], f, B[:,range_a:range_b], cmap='gray')
    plt.title('B binarized')
    plt.show()

In [ ]:
# -- median filter
Bm = ndimage.median_filter(B, size=(3, 3))

# -- kernels for morphological operations
kernel_rect  = np.ones((4,2), np.uint8)
kernel_line1 = np.ones((4,1), np.uint8)
kernel_line2 = np.ones((5,1), np.uint8)

# -- morphological operations
erode11  = cv2.erode(Bm, kernel_line1, iterations=1)
dilate12 = cv2.dilate(erode11, kernel_rect, iterations=1)
dilate13 = cv2.dilate(dilate12, kernel_line2, iterations=1)
erode14  = cv2.erode(dilate13, kernel_line1, iterations=2)

if args:
    plt.pcolormesh(t[range_a:range_b], f, B[:,range_a:range_b], cmap='gray')
    plt.title('B binarized')
    plt.show()
    plt.pcolormesh(t[range_a:range_b], f, Bm[:,range_a:range_b], cmap='gray')
    plt.title('B binarized + median filter')
    plt.show()
    plt.pcolormesh(t[range_a:range_b], f, erode11[:,range_a:range_b], cmap='gray')
    plt.title('+ erode (4,1)')
    plt.show()
    plt.pcolormesh(t[range_a:range_b], f, dilate12[:,range_a:range_b], cmap='gray')
    plt.title('+ dilate (4,2)')
    plt.show()
    plt.pcolormesh(t[range_a:range_b], f, dilate13[:,range_a:range_b], cmap='gray')
    plt.title('+ dilate (5,1)')
    plt.show()
    plt.pcolormesh(t[range_a:range_b], f, erode14[:,range_a:range_b], cmap='gray')
    plt.title('+ erode (4,1)')
    plt.show()

In [ ]:
timeAConnectedComponents = time()
connectivity = 4
num_cc, output, stats, centroids = cv2.connectedComponentsWithStats(erode14, connectivity, cv2.CV_32S)

# -- remove background stats
num_cc = num_cc - 1
areas  = stats[1:,4]
# nr     = np.arange(num_cc)
# ranked = sorted(zip(areas,nr))

# -- filtered connected components placeholder
grain = np.zeros((output.shape))

# -- threshold connected components by minimum area
min_area = 20
for i in range(0, num_cc):
    if areas[i] >= min_area:
        grain[output == i + 1] = 255
timeBConnectedComponents = time()

# -- one more opening to make sure segmentation covers *at least* the real area
grain        = grain.astype(np.uint8)
kernel_cross = np.array([[0, 1, 0],
                         [1, 1, 1],
                         [0, 1, 0]], dtype=np.uint8)
kernel_line3 = np.ones((1,3), dtype=np.uint8)
grain        = cv2.dilate(grain, kernel_line3, iterations=1)

# -- rescale to save spectrograms in 8bit grayscale
dtype      = np.uint8
Pxx_scaled = exposure.rescale_intensity(Pxx, in_range='image', out_range=dtype)

timeARegionProps = time()
labels = measure.label(erode11, background=0)

props  = measure.regionprops(labels, intensity_image=Pxx, cache=True, coordinates='rc')
props  = sorted(props, key=lambda p: np.min(p.coords[:,1]), reverse=False)
timeBRegionProps = time()
if args:
#     plt.subplot(311)
    plt.pcolormesh(t[range_a:range_b], f, Pxx[:,range_a:range_b], cmap='gray')
    plt.title('Pxx')
    plt.show()
#     plt.subplot(312)
    plt.pcolormesh(t[range_a:range_b], f, grain[:,range_a:range_b], cmap='gray')
    plt.title('grain')
    plt.show()
#     plt.subplot(313)
    plt.pcolormesh(t[range_a:range_b], f, labels[:,range_a:range_b], cmap='magma')
    plt.title('Labels')
#     plt.show()

# if args:
#     plt.pcolormesh(t[range_a:range_b], f, B[:,range_a:range_b], cmap='gray')
#     # plt.pcolormesh(t, f, B, cmap='gray')
#     plt.title('B binarized')
#     plt.show()
#     plt.pcolormesh(t[range_a:range_b], f, grain[:,range_a:range_b], cmap='gray')
#     # plt.pcolormesh(t, f, grain, cmap='gray')
#     plt.title('B after image processing steps')
#     plt.show()
#     # -- get spectrogram area using the segmentation mask
#     B_masked = Pxx_scaled * (((255 - grain) > 0) * 1)
#     plt.pcolormesh(t[range_a:range_b], f, Pxx[:,range_a:range_b], cmap='gray')
#     # plt.pcolormesh(t, f, Pxx, cmap='gray')
#     plt.title('Original spectrogram')
#     plt.show()
#     plt.pcolormesh(t[range_a:range_b], f, B_masked[:,range_a:range_b], cmap='gray')
#     # plt.pcolormesh(t, f, B_masked, cmap='gray')
#     plt.title('Spectrogram + identified vocalizations')
#     plt.show()